# 提取全局特征（GTA)

In [1]:
# 提取局部结构特征第一步
import os

structure_path = './data/structures/GTA/'
folders = os.listdir(structure_path)

f = open('./Protein_extract_feature_step1.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        f.write(f"python ./Extract_feature_step1_diffpool2.py {file} {folder} 6 6 GTA\n")
f.close()

# nohup parallel --jobs 30 < Protein_extract_feature_step1.sh > Feature_Step_1.out 2>&1 &


# 提取局部特征（GTA）

In [3]:
# 提取局部结构特征第二步
import os

structure_path = './data/structures/GTA/'
folders = os.listdir(structure_path)

sample_redies_udp = 6
sample_redies_sugar = 6

step = 1
epoch = 1
f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        if step % 15 == 0:
            f.write(f"taskset -c {step} python ./Extract_feature_step2_diffpool2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTA\n")
            f.close()
            epoch += 1
            f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
            step = 1
            continue
        f.write(f"taskset -c {step} python ./Extract_feature_step2_diffpool2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTA\n")
        step = step + 1
f.close()

f = open(f'./Extract_feature_step2_AAArun.sh', 'w')
for e in range(epoch):
    f.write(f"parallel --jobs 15 < Extract_feature_step2_epoch{e+1}.sh\n")
f.close()

# nohup bash Extract_feature_step2_AAArun.sh &


# 生成数据集 - 排除NaN的数据（GTA）

In [1]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
    'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
    'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
    'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTA/'
    dl_data_path = './data/dl_data/GTA/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTA_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[], 'nova':[]}
    df = pd.read_excel(f'./data/cluster/GTA/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']
    nova_process_path = process_path['nova']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')
    os.makedirs(f'{dl_data_path}/nova/')
    os.makedirs(f'{dl_data_path}/nova/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')
    make_database(nova_process_path, 'nova')

100%|██████████| 475/475 [00:02<00:00, 189.33it/s]


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 生成数据集 - 排除NaN的数据（GTA）- 使用全部数据

In [1]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
    'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
    'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
    'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTA/'
    dl_data_path = './data/dl_data/GTA_alldata/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTA_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTA_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')

100%|██████████| 1305/1305 [00:07<00:00, 184.29it/s]


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 生成数据集 - 排除NaN的数据（GTA）- 使用全部数据 - 包含item_id标签

In [1]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
    'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
    'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
    'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTA/'
    dl_data_path = './data/dl_data/GTA_alldata_id/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTA_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTA_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_itemID = open(f'{dl_data_path}/{data_type}/GTmining_itemID.txt', 'w')  # 新增：itemID文件句柄
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # 获取当前文件的唯一标识作为item ID
            item_id = file.split('/')[-1].split('.npy')[0]
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ 新增：f_itemID的数据处理 +++++
            # 每个图对应一行item ID
            f_itemID.write(f"{item_id}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_itemID.close()  # 新增：关闭itemID文件
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')

/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1316/1316 [00:09<00:00, 139.93it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1312/1312 [00:05<00:00, 254.51it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1203/1203 [00:04<00:00, 249.12it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1338

# 生成数据集 - 排除NaN的数据（GTA）- 全部数据 - item_id - 消融实验

In [1]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
    'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
    'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
    'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [ ]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

fold_num = 5
abl_type = 'hphob' # xyz, si, hbond, charge, hphob

# 获取有哪些local结构
storage_path = './data/local_features/GTA/'
dl_data_path = f'./data/dl_data/GTA_alldata_id_abl_{abl_type}/'
dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

folders = os.listdir(storage_path)
local_dict = {}
for folder in folders:
    npy_path = os.path.join(storage_path, folder)
    local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

# 获取活性标签
df = pd.read_excel('./data/GTA_training_data.xlsx')
activate_dict = {}
for i in range(0,df.shape[0]):
    if '[-]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
    elif '[*]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
    else:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

# 生成文件列表
process_path = {'train':[], 'validation':[], 'test':[]}
df = pd.read_excel(f'./data/cluster/GTA_alldata/dataseat_split_{fold_num}.xlsx')
for i in range(0,df.shape[0]):
    if df['GenBank'][i] in local_dict[df['Family'][i]]:
        if df['GenBank'][i] in skip_list:
            continue
        npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
        process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
train_process_path = process_path['train']
validation_process_path = process_path['validation']
test_process_path = process_path['test']

# 管理文件夹
if not os.path.isdir(dl_data_path):
    os.makedirs(dl_data_path, exist_ok=True)
else:
    shutil.rmtree(dl_data_path)
    os.makedirs(dl_data_path, exist_ok=True)
os.makedirs(f'{dl_data_path}/train/')
os.makedirs(f'{dl_data_path}/train/trace_file/')
os.makedirs(f'{dl_data_path}/validation/')
os.makedirs(f'{dl_data_path}/validation/trace_file/')
os.makedirs(f'{dl_data_path}/test/')
os.makedirs(f'{dl_data_path}/test/trace_file/')

# ==================== 生成数据 ====================
def make_database(process_path: list, data_type: str):
    f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
    f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
    f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
    f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
    f_itemID = open(f'{dl_data_path}/{data_type}/GTmining_itemID.txt', 'w')  # 新增：itemID文件句柄
    f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
    edge_max = -1
    graph_idx = -1
    # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
    for file in tqdm(process_path):
        # +++++ 一次性读取npy文件 +++++
        try:
            input_dict = np.load(file, allow_pickle=True)
        except:
            print(f"wrong {file}")
            continue
        input_dict = input_dict[()]
        if len(input_dict['edges']) <= 100:
            # 用来检查不正确的局部网格
            print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
            continue
        # 获取当前文件的唯一标识作为item ID
        item_id = file.split('/')[-1].split('.npy')[0]
        # +++++ f_A的数据处理 +++++
        edges = input_dict['edges']
        edges += (edge_max +1)
        for edge in edges:
            f_A.write(f"{edge[0]}, {edge[1]}\n")
        edge_max = np.max(edges)
        # +++++ f_graph_indicator的数据处理 +++++
        graph_idx += 1
        xyzs = input_dict['xyz']
        for i in range(0, xyzs.shape[0]):
            f_graph_indicator.write(f"{graph_idx}\n")
        # +++++ f_graph_labels的数据处理 +++++
        group_name = file.split('/')[-1].split('.npy')[0]
        f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
        # +++++ 新增：f_itemID的数据处理 +++++
        # 每个图对应一行item ID
        f_itemID.write(f"{item_id}\n")
        # +++++ f_node_attributes的数据处理 +++++
        xyzs = input_dict['xyz']
        sis = input_dict['si']
        hbonds = input_dict['hbond']
        charges = input_dict['charge']
        hphobs = input_dict['hphob']
        trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
        f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
        for i in range(0, xyzs.shape[0]):
            # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
            # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
            # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
            x = xyzs[i][0]
            y = xyzs[i][1]
            z = xyzs[i][2]
            f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
            # f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            if abl_type == 'xyz':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            elif abl_type == 'si':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, hbonds[i][0], charges[i][0], hphobs[i][0]))
            elif abl_type == 'hbond':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], charges[i][0], hphobs[i][0]))
            elif abl_type == 'charge':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], hphobs[i][0]))
            elif abl_type == 'hphob':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0]))
            else:
                raise ValueError("Invalid abl_type. Choose from 'xyz', 'si', 'hbond', 'charge', 'hphob'.")
        f_trace_file.close()
        f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
        temp_correspond_information = ''
        temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                            round(xyzs[0][1], 5),
                                                                                                            round(xyzs[0][2], 5),
                                                                                                            round(sis[0][0], 5),
                                                                                                            round(hbonds[0][0], 5),
                                                                                                            round(charges[0][0], 5),
                                                                                                            round(hphobs[0][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                round(xyzs[1][1], 5),
                                                                                                                round(xyzs[1][2], 5),
                                                                                                                round(sis[1][0], 5),
                                                                                                                round(hbonds[1][0], 5),
                                                                                                                round(charges[1][0], 5),
                                                                                                                round(hphobs[1][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                round(xyzs[2][1], 5),
                                                                                                                round(xyzs[2][2], 5),
                                                                                                                round(sis[2][0], 5),
                                                                                                                round(hbonds[2][0], 5),
                                                                                                                round(charges[2][0], 5),
                                                                                                                round(hphobs[2][0], 5))
        if temp_correspond_information in skip_correspond_information:
            temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
        f_correspond_information.write(temp_correspond_information)

    f_A.close()
    f_graph_indicator.close()
    f_graph_labels.close()
    f_itemID.close()  # 新增：关闭itemID文件
    f_node_attributes.close()
    f_correspond_information.close()

make_database(train_process_path, 'train')
make_database(validation_process_path, 'validation')
make_database(test_process_path, 'test')

/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1289/1289 [00:04<00:00, 268.32it/s]


# 生成数据集 - 排除NaN的数据（GTA）- 使用全部数据 - 包含item_id标签 - 0.6分辨率

In [ ]:
# skip_list = [
#     'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
#     'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
#     'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
#     'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
#     'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
#     'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
#     'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
#     'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
# ]

# skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
#     '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
#     '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
#     '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
#     '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
#     '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
#     '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
#     '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
#     '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
# }

# skip_correspond_information = list(skip_correspond_information_dict.keys())

In [8]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',  # 无法计算特征
    'GT2_UAT38450_1', 'GT2_QMK65633_1', 'GT27_WHX89188_1', 'GT2_QLS93953_1', 'GT2_UXD96330_1', 'GT2_AXO40002_1', 'GT2_UOP56892_1',  # 无法前向传播
    'GT2_UAY64771_1', 'GT2_QGN40148_1', 'GT2_AKG29453_1', 'GT2_AAX64275_1', 'GT2_QQH93332_1', 'GT8_CAG1855473_1', 'GT2_AMW54931_1',  # 无法前向传播
    'GT2_CCO56775_1', 'GT2_AAF82801_1', 'GT2_AOF92917_1', 'GT2_UIL51672_1', 'GT2_AGQ59923_1', 'GT15_WBF16128_1', 'GT2_WKW43796_1',  # 无法前向传播
    'GT2_APJ59072_1', 'GT2_QQH80255_1', 'GT2_QQX53275_1', 'GT8_ACD48839_1', 'GT2_ALQ52379_1', 'GT2_AUV03617_1', 'GT2_QKJ86734_1',  # 无法前向传播
    'GT12_ALR88580_1', 'GT62_UJO12658_1', 'GT8_AUA41471_1', 'GT8_UZL98811_1', 'GT2_QYH13917_1', 'GT81_VEG37745_1', 'GT7_AFP00606_1',  # 无法前向传播
    'GT2_AID25391_1', 'GT2_QQJ86676_1'  # 无法前向传播
    # '', '', '', '', '', '', '',  # 无法前向传播
]

skip_correspond_information_dict = {}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [10]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data2/local_features/GTA/'
    dl_data_path = './data2/dl_data/GTA_alldata_id/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTA_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTA_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_itemID = open(f'{dl_data_path}/{data_type}/GTmining_itemID.txt', 'w')  # 新增：itemID文件句柄
        # f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # 获取当前文件的唯一标识作为item ID
            item_id = file.split('/')[-1].split('.npy')[0]
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ 新增：f_itemID的数据处理 +++++
            # 每个图对应一行item ID
            f_itemID.write(f"{item_id}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            # trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            # f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                # f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            # f_trace_file.close()
            # f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            # temp_correspond_information = ''
            # temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
            #                                                                                                     round(xyzs[0][1], 5),
            #                                                                                                     round(xyzs[0][2], 5),
            #                                                                                                     round(sis[0][0], 5),
            #                                                                                                     round(hbonds[0][0], 5),
            #                                                                                                     round(charges[0][0], 5),
            #                                                                                                     round(hphobs[0][0], 5))
            # temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
            #                                                                                                         round(xyzs[1][1], 5),
            #                                                                                                         round(xyzs[1][2], 5),
            #                                                                                                         round(sis[1][0], 5),
            #                                                                                                         round(hbonds[1][0], 5),
            #                                                                                                         round(charges[1][0], 5),
            #                                                                                                         round(hphobs[1][0], 5))
            # temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
            #                                                                                                         round(xyzs[2][1], 5),
            #                                                                                                         round(xyzs[2][2], 5),
            #                                                                                                         round(sis[2][0], 5),
            #                                                                                                         round(hbonds[2][0], 5),
            #                                                                                                         round(charges[2][0], 5),
            #                                                                                                         round(hphobs[2][0], 5))
            # if temp_correspond_information in skip_correspond_information:
            #     temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            # f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_itemID.close()  # 新增：关闭itemID文件
        f_node_attributes.close()
        # f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')

    # break

/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1307/1307 [00:10<00:00, 122.01it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1311/1311 [00:10<00:00, 119.80it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1195/1195 [00:10<00:00, 117.25it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 1327

# 提取全局特征（GTB)

In [24]:
# 提取局部结构特征第一步
import os

structure_path = './data/structures/GTB/'
folders = os.listdir(structure_path)

f = open('./Protein_extract_feature_step1.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        f.write(f"python ./Extract_feature_step1.py {file} {folder} 6 6 GTB\n")
f.close()

# nohup parallel --jobs 30 < Protein_extract_feature_step1.sh > Feature_Step_1.out 2>&1 &


# 提取局部特征（GTB）

In [1]:
# 提取局部结构特征第二步
import os

structure_path = './data/structures/GTB/'
folders = os.listdir(structure_path)

sample_redies_udp = 6
sample_redies_sugar = 6

step = 1
epoch = 1
f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        if step % 40 == 0:
            f.write(f"taskset -c {step} python ./Extract_feature_step2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTB\n")
            f.close()
            epoch += 1
            f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
            step = 1
            continue
        f.write(f"taskset -c {step} python ./Extract_feature_step2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTB\n")
        step = step + 1
f.close()

f = open(f'./Extract_feature_step2_AAArun.sh', 'w')
for e in range(epoch):
    f.write(f"parallel --jobs 40 < Extract_feature_step2_epoch{e+1}.sh\n")
f.close()

# nohup bash Extract_feature_step2_AAArun.sh &


# 生成数据集 - 排除NaN的数据（GTB）

In [1]:
skip_list = [
    'GT35_AHY09627_1', 'GT19_BBB60788_1',                         # 不能前向传播，含NaN，fold1，test
    'GT10_AAW32622_1', 'GT106_CAB4104418_1', 'GT35_QKX50649_1',   # 不能前向传播，含NaN，fold1，validation
    'GT19_WYA60860_1', 'GT56_UXY10792_1', 'GT35_QIX92616_1',      # 不能前向传播，含NaN，fold1，validation
    'GT30_QNF15040_1', 'GT35_WWU74798_1',                         # 不能前向传播，含NaN，fold1，validation
    'GT70_APO94310_1', 'GT70_QTK46120_1', 'GT5_AGB47314_1',       # 不能前向传播，含NaN，fold1，train
    'GT19_AZA48355_1', 'GT104_UXB11094_1', 'GT30_QOV30031_1',     # 不能前向传播，含NaN，fold1，train
    'GT4_UDD40513_1', 'GT4_AYU93913_1', 'GT56_QHA87116_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_BCO86114_1', 'GT30_UTV85551_1', 'GT5_CAI9087173_1',      # 不能前向传播，含NaN，fold1，train
    'GT20_QLV68278_1', 'GT4_UHD26605_1', 'GT4_BBA87692_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_QGS26923_1', 'GT19_ABQ29269_1', 'GT30_QQC33163_1',       # 不能前向传播，含NaN，fold1，train
    'GT30_UWQ44439_1', 'GT4_CAI34053_2', 'GT4_ALI28300_1',        # 不能前向传播，含NaN，fold1，train
    'GT5_WQB98038_1', 'GT4_CCC19979_1', 'GT30_ARG39747_1',        # 不能前向传播，含NaN，fold1，train
    'GT19_XAU06904_1', 'GT35_ACM59680_1', 'GT56_ALB53317_1',      # 不能前向传播，含NaN，fold1，train
    'GT113_VFH88248_1'                                            # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '5.98, -7.13, -3.93, -0.43, 0.86, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n':
    '5.98, -7.13, -3.93, -0.43, 0.87, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n',               # 不能索引名称，数字错误，fold1， train
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.61, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n':
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.62, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n',     # 不能索引名称，数字错误，fold1， train
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.05, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n':
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.06, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.65, 0.00, -0.02, -0.78\n':
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.64, 0.00, -0.02, -0.78\n',           # 不能索引名称，数字错误，fold1， train
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.33, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n':
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.34, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.08, 0.55, -0.29\n':
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.09, 0.55, -0.29\n',         # 不能索引名称，数字错误，fold1， train
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.02, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n':
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.01, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n',            # 不能索引名称，数字错误，fold1， train
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.10, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n':
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.11, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n',            # 不能索引名称，数字错误，fold1， train
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.17, 0.00, 0.56, -0.87\n':
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.16, 0.00, 0.56, -0.87\n'               # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                    'dTDP-Rha': 8, 'Other': 9}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTB/'
    dl_data_path = './data/dl_data/GTB/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTB_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[], 'nova':[]}
    df = pd.read_excel(f'./data/cluster/GTB/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']
    nova_process_path = process_path['nova']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')
    os.makedirs(f'{dl_data_path}/nova/')
    os.makedirs(f'{dl_data_path}/nova/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')
    make_database(nova_process_path, 'nova')

100%|██████████| 2194/2194 [00:13<00:00, 159.80it/s]


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 生成数据集 - 排除NaN的数据（GTB）- 使用全部数据

In [1]:
skip_list = [
    'GT35_AHY09627_1', 'GT19_BBB60788_1',                         # 不能前向传播，含NaN，fold1，test
    'GT10_AAW32622_1', 'GT106_CAB4104418_1', 'GT35_QKX50649_1',   # 不能前向传播，含NaN，fold1，validation
    'GT19_WYA60860_1', 'GT56_UXY10792_1', 'GT35_QIX92616_1',      # 不能前向传播，含NaN，fold1，validation
    'GT30_QNF15040_1', 'GT35_WWU74798_1',                         # 不能前向传播，含NaN，fold1，validation
    'GT70_APO94310_1', 'GT70_QTK46120_1', 'GT5_AGB47314_1',       # 不能前向传播，含NaN，fold1，train
    'GT19_AZA48355_1', 'GT104_UXB11094_1', 'GT30_QOV30031_1',     # 不能前向传播，含NaN，fold1，train
    'GT4_UDD40513_1', 'GT4_AYU93913_1', 'GT56_QHA87116_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_BCO86114_1', 'GT30_UTV85551_1', 'GT5_CAI9087173_1',      # 不能前向传播，含NaN，fold1，train
    'GT20_QLV68278_1', 'GT4_UHD26605_1', 'GT4_BBA87692_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_QGS26923_1', 'GT19_ABQ29269_1', 'GT30_QQC33163_1',       # 不能前向传播，含NaN，fold1，train
    'GT30_UWQ44439_1', 'GT4_CAI34053_2', 'GT4_ALI28300_1',        # 不能前向传播，含NaN，fold1，train
    'GT5_WQB98038_1', 'GT4_CCC19979_1', 'GT30_ARG39747_1',        # 不能前向传播，含NaN，fold1，train
    'GT19_XAU06904_1', 'GT35_ACM59680_1', 'GT56_ALB53317_1',      # 不能前向传播，含NaN，fold1，train
    'GT113_VFH88248_1'                                            # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '5.98, -7.13, -3.93, -0.43, 0.86, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n':
    '5.98, -7.13, -3.93, -0.43, 0.87, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n',               # 不能索引名称，数字错误，fold1， train
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.61, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n':
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.62, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n',     # 不能索引名称，数字错误，fold1， train
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.05, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n':
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.06, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.65, 0.00, -0.02, -0.78\n':
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.64, 0.00, -0.02, -0.78\n',           # 不能索引名称，数字错误，fold1， train
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.33, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n':
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.34, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.08, 0.55, -0.29\n':
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.09, 0.55, -0.29\n',         # 不能索引名称，数字错误，fold1， train
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.02, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n':
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.01, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n',            # 不能索引名称，数字错误，fold1， train
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.10, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n':
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.11, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n',            # 不能索引名称，数字错误，fold1， train
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.17, 0.00, 0.56, -0.87\n':
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.16, 0.00, 0.56, -0.87\n'               # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                    'dTDP-Rha': 8, 'Other': 9}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTB/'
    dl_data_path = './data/dl_data/GTB_alldata/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTB_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTB_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']


    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')


100%|██████████| 2109/2109 [00:27<00:00, 75.60it/s] 


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 生成数据集 - 排除NaN的数据（GTB）- 使用全部数据 - 包含item_id标签

In [1]:
skip_list = [
    'GT35_AHY09627_1', 'GT19_BBB60788_1',                         # 不能前向传播，含NaN，fold1，test
    'GT10_AAW32622_1', 'GT106_CAB4104418_1', 'GT35_QKX50649_1',   # 不能前向传播，含NaN，fold1，validation
    'GT19_WYA60860_1', 'GT56_UXY10792_1', 'GT35_QIX92616_1',      # 不能前向传播，含NaN，fold1，validation
    'GT30_QNF15040_1', 'GT35_WWU74798_1',                         # 不能前向传播，含NaN，fold1，validation
    'GT70_APO94310_1', 'GT70_QTK46120_1', 'GT5_AGB47314_1',       # 不能前向传播，含NaN，fold1，train
    'GT19_AZA48355_1', 'GT104_UXB11094_1', 'GT30_QOV30031_1',     # 不能前向传播，含NaN，fold1，train
    'GT4_UDD40513_1', 'GT4_AYU93913_1', 'GT56_QHA87116_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_BCO86114_1', 'GT30_UTV85551_1', 'GT5_CAI9087173_1',      # 不能前向传播，含NaN，fold1，train
    'GT20_QLV68278_1', 'GT4_UHD26605_1', 'GT4_BBA87692_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_QGS26923_1', 'GT19_ABQ29269_1', 'GT30_QQC33163_1',       # 不能前向传播，含NaN，fold1，train
    'GT30_UWQ44439_1', 'GT4_CAI34053_2', 'GT4_ALI28300_1',        # 不能前向传播，含NaN，fold1，train
    'GT5_WQB98038_1', 'GT4_CCC19979_1', 'GT30_ARG39747_1',        # 不能前向传播，含NaN，fold1，train
    'GT19_XAU06904_1', 'GT35_ACM59680_1', 'GT56_ALB53317_1',      # 不能前向传播，含NaN，fold1，train
    'GT113_VFH88248_1'                                            # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '5.98, -7.13, -3.93, -0.43, 0.86, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n':
    '5.98, -7.13, -3.93, -0.43, 0.87, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n',               # 不能索引名称，数字错误，fold1， train
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.61, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n':
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.62, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n',     # 不能索引名称，数字错误，fold1， train
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.05, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n':
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.06, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.65, 0.00, -0.02, -0.78\n':
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.64, 0.00, -0.02, -0.78\n',           # 不能索引名称，数字错误，fold1， train
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.33, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n':
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.34, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.08, 0.55, -0.29\n':
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.09, 0.55, -0.29\n',         # 不能索引名称，数字错误，fold1， train
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.02, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n':
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.01, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n',            # 不能索引名称，数字错误，fold1， train
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.10, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n':
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.11, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n',            # 不能索引名称，数字错误，fold1， train
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.17, 0.00, 0.56, -0.87\n':
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.16, 0.00, 0.56, -0.87\n'               # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                    'dTDP-Rha': 8, 'Other': 9}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTB/'
    dl_data_path = './data/dl_data/GTB_alldata_id/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTB_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTB_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']


    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_itemID = open(f'{dl_data_path}/{data_type}/GTmining_itemID.txt', 'w')  # 新增：itemID文件句柄
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # 获取当前文件的唯一标识作为item ID
            item_id = file.split('/')[-1].split('.npy')[0]
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ 新增：f_itemID的数据处理 +++++
            # 每个图对应一行item ID
            f_itemID.write(f"{item_id}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_itemID.close()  # 新增：关闭itemID文件
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')


/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 2168/2168 [00:33<00:00, 65.18it/s] 
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 2044/2044 [00:08<00:00, 233.01it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 2117/2117 [00:09<00:00, 216.69it/s]
/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 2131

# 生成数据集 - 排除NaN的数据（GTB）- 全部数据 - item_id - 消融实验

In [2]:
skip_list = [
    'GT35_AHY09627_1', 'GT19_BBB60788_1',                         # 不能前向传播，含NaN，fold1，test
    'GT10_AAW32622_1', 'GT106_CAB4104418_1', 'GT35_QKX50649_1',   # 不能前向传播，含NaN，fold1，validation
    'GT19_WYA60860_1', 'GT56_UXY10792_1', 'GT35_QIX92616_1',      # 不能前向传播，含NaN，fold1，validation
    'GT30_QNF15040_1', 'GT35_WWU74798_1',                         # 不能前向传播，含NaN，fold1，validation
    'GT70_APO94310_1', 'GT70_QTK46120_1', 'GT5_AGB47314_1',       # 不能前向传播，含NaN，fold1，train
    'GT19_AZA48355_1', 'GT104_UXB11094_1', 'GT30_QOV30031_1',     # 不能前向传播，含NaN，fold1，train
    'GT4_UDD40513_1', 'GT4_AYU93913_1', 'GT56_QHA87116_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_BCO86114_1', 'GT30_UTV85551_1', 'GT5_CAI9087173_1',      # 不能前向传播，含NaN，fold1，train
    'GT20_QLV68278_1', 'GT4_UHD26605_1', 'GT4_BBA87692_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_QGS26923_1', 'GT19_ABQ29269_1', 'GT30_QQC33163_1',       # 不能前向传播，含NaN，fold1，train
    'GT30_UWQ44439_1', 'GT4_CAI34053_2', 'GT4_ALI28300_1',        # 不能前向传播，含NaN，fold1，train
    'GT5_WQB98038_1', 'GT4_CCC19979_1', 'GT30_ARG39747_1',        # 不能前向传播，含NaN，fold1，train
    'GT19_XAU06904_1', 'GT35_ACM59680_1', 'GT56_ALB53317_1',      # 不能前向传播，含NaN，fold1，train
    'GT113_VFH88248_1'                                            # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '5.98, -7.13, -3.93, -0.43, 0.86, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n':
    '5.98, -7.13, -3.93, -0.43, 0.87, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n',               # 不能索引名称，数字错误，fold1， train
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.61, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n':
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.62, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n',     # 不能索引名称，数字错误，fold1， train
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.05, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n':
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.06, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.65, 0.00, -0.02, -0.78\n':
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.64, 0.00, -0.02, -0.78\n',           # 不能索引名称，数字错误，fold1， train
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.33, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n':
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.34, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.08, 0.55, -0.29\n':
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.09, 0.55, -0.29\n',         # 不能索引名称，数字错误，fold1， train
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.02, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n':
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.01, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n',            # 不能索引名称，数字错误，fold1， train
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.10, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n':
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.11, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n',            # 不能索引名称，数字错误，fold1， train
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.17, 0.00, 0.56, -0.87\n':
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.16, 0.00, 0.56, -0.87\n'               # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [7]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                    'dTDP-Rha': 8, 'Other': 9}

fold_num = 5
abl_type = 'hphob' # xyz, si, hbond, charge, hphob

# 获取有哪些local结构
storage_path = './data/local_features/GTB/'
dl_data_path = f'./data/dl_data/GTB_alldata_id_abl_{abl_type}/'
dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

folders = os.listdir(storage_path)
local_dict = {}
for folder in folders:
    npy_path = os.path.join(storage_path, folder)
    local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

# 获取活性标签
df = pd.read_excel('./data/GTB_training_data.xlsx')
activate_dict = {}
for i in range(0,df.shape[0]):
    if '[-]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
    elif '[*]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
    else:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

# 生成文件列表
process_path = {'train':[], 'validation':[], 'test':[]}
df = pd.read_excel(f'./data/cluster/GTB_alldata/dataseat_split_{fold_num}.xlsx')
for i in range(0,df.shape[0]):
    if df['GenBank'][i] in local_dict[df['Family'][i]]:
        if df['GenBank'][i] in skip_list:
            continue
        npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
        process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
train_process_path = process_path['train']
validation_process_path = process_path['validation']
test_process_path = process_path['test']


# 管理文件夹
if not os.path.isdir(dl_data_path):
    os.makedirs(dl_data_path, exist_ok=True)
else:
    shutil.rmtree(dl_data_path)
    os.makedirs(dl_data_path, exist_ok=True)
os.makedirs(f'{dl_data_path}/train/')
os.makedirs(f'{dl_data_path}/train/trace_file/')
os.makedirs(f'{dl_data_path}/validation/')
os.makedirs(f'{dl_data_path}/validation/trace_file/')
os.makedirs(f'{dl_data_path}/test/')
os.makedirs(f'{dl_data_path}/test/trace_file/')

# ==================== 生成数据 ====================
def make_database(process_path: list, data_type: str):
    f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
    f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
    f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
    f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
    f_itemID = open(f'{dl_data_path}/{data_type}/GTmining_itemID.txt', 'w')  # 新增：itemID文件句柄
    f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
    edge_max = -1
    graph_idx = -1
    # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
    for file in tqdm(process_path):
        # +++++ 一次性读取npy文件 +++++
        try:
            input_dict = np.load(file, allow_pickle=True)
        except:
            print(f"wrong {file}")
            continue
        input_dict = input_dict[()]
        if len(input_dict['edges']) <= 100:
            # 用来检查不正确的局部网格
            print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
            continue
        # 获取当前文件的唯一标识作为item ID
        item_id = file.split('/')[-1].split('.npy')[0]
        # +++++ f_A的数据处理 +++++
        edges = input_dict['edges']
        edges += (edge_max +1)
        for edge in edges:
            f_A.write(f"{edge[0]}, {edge[1]}\n")
        edge_max = np.max(edges)
        # +++++ f_graph_indicator的数据处理 +++++
        graph_idx += 1
        xyzs = input_dict['xyz']
        for i in range(0, xyzs.shape[0]):
            f_graph_indicator.write(f"{graph_idx}\n")
        # +++++ f_graph_labels的数据处理 +++++
        group_name = file.split('/')[-1].split('.npy')[0]
        f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
        # +++++ 新增：f_itemID的数据处理 +++++
        # 每个图对应一行item ID
        f_itemID.write(f"{item_id}\n")
        # +++++ f_node_attributes的数据处理 +++++
        xyzs = input_dict['xyz']
        sis = input_dict['si']
        hbonds = input_dict['hbond']
        charges = input_dict['charge']
        hphobs = input_dict['hphob']
        trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
        f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
        for i in range(0, xyzs.shape[0]):
            # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
            # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
            # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
            x = xyzs[i][0]
            y = xyzs[i][1]
            z = xyzs[i][2]
            f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
            # f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            if abl_type == 'xyz':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            elif abl_type == 'si':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, hbonds[i][0], charges[i][0], hphobs[i][0]))
            elif abl_type == 'hbond':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], charges[i][0], hphobs[i][0]))
            elif abl_type == 'charge':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], hphobs[i][0]))
            elif abl_type == 'hphob':
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0]))
            else:
                raise ValueError("Invalid abl_type. Choose from 'xyz', 'si', 'hbond', 'charge', 'hphob'.")
        f_trace_file.close()
        f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
        temp_correspond_information = ''
        temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                            round(xyzs[0][1], 5),
                                                                                                            round(xyzs[0][2], 5),
                                                                                                            round(sis[0][0], 5),
                                                                                                            round(hbonds[0][0], 5),
                                                                                                            round(charges[0][0], 5),
                                                                                                            round(hphobs[0][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                round(xyzs[1][1], 5),
                                                                                                                round(xyzs[1][2], 5),
                                                                                                                round(sis[1][0], 5),
                                                                                                                round(hbonds[1][0], 5),
                                                                                                                round(charges[1][0], 5),
                                                                                                                round(hphobs[1][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                round(xyzs[2][1], 5),
                                                                                                                round(xyzs[2][2], 5),
                                                                                                                round(sis[2][0], 5),
                                                                                                                round(hbonds[2][0], 5),
                                                                                                                round(charges[2][0], 5),
                                                                                                                round(hphobs[2][0], 5))
        if temp_correspond_information in skip_correspond_information:
            temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
        f_correspond_information.write(temp_correspond_information)

    f_A.close()
    f_graph_indicator.close()
    f_graph_labels.close()
    f_itemID.close()  # 新增：关闭itemID文件
    f_node_attributes.close()
    f_correspond_information.close()

make_database(train_process_path, 'train')
make_database(validation_process_path, 'validation')
make_database(test_process_path, 'test')


/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
100%|██████████| 2133/2133 [00:09<00:00, 229.55it/s]


# 计算并展示活性架构景观

In [1]:
# # 摸索以一个结构为模板探索特征提取方法
from MaSIF.protonate import protonate
from MaSIF.computeMSMS import computeMSMS
from MaSIF.computeCharges import computeCharges, assignChargesToNewMesh
from MaSIF.computeHydrophobicity import computeHydrophobicity
from MaSIF.fixmesh import fix_mesh
from MaSIF.masif_opts import masif_opts
from MaSIF.compute_normal import compute_normal
from MaSIF.computeAPBS import computeAPBS
from MaSIF.save_ply import save_ply
from MaSIF.read_data_from_surface import read_data_from_surface
import numpy as np
from Bio.PDB import *
import pymesh
from sklearn.neighbors import KDTree
import os

pdb_filename = '2vfz_align.pdb'
sample_redies_udp = 6.0
sample_redies_sugar = 6.0
collapse_type = 'GTA'

pdb_path = f'./ref/'
temp_path = f'./ref/'
storage_path = f'./ref/'

# 构建基础文件名
original_file = os.path.join(pdb_path, pdb_filename)
protonate_file = os.path.join(temp_path, pdb_filename)
ply_filename = os.path.join(temp_path, pdb_filename.replace('.pdb', '.ply'))
storage_filename = os.path.join(storage_path, pdb_filename.replace('.pdb', '.npy'))

# Protonated the pdb structure. 
protonate(original_file, protonate_file)

# Compute MSMS of surface w/hydrogens.
vertices1, faces1, normals1, names1, areas1 = computeMSMS(protonate_file, protonate=True)
# Compute "charged" vertices
vertex_hbond = computeCharges(protonate_file, vertices1, names1)
# For each surface residue, assign the hydrophobicity of its amino acid. 
vertex_hphobicity = computeHydrophobicity(names1)

vertices2 = vertices1
faces2 = faces1

# Fix the mesh.
mesh = pymesh.form_mesh(vertices2, faces2)
mesh_original = mesh # ==========测试
regular_mesh = fix_mesh(mesh, masif_opts['mesh_res'])
mesh_fixed = regular_mesh # ==========测试
# Compute the normals
vertex_normal = compute_normal(regular_mesh.vertices, regular_mesh.faces)

# Assign charges on new vertices based on charges of old vertices (nearest neighbor)
vertex_hbond = assignChargesToNewMesh(regular_mesh.vertices, vertices1, vertex_hbond, masif_opts)
vertex_hphobicity = assignChargesToNewMesh(regular_mesh.vertices, vertices1, vertex_hphobicity, masif_opts)

vertex_charges = computeAPBS(regular_mesh.vertices, protonate_file, protonate_file.split('.pdb')[0])

iface = np.zeros(len(regular_mesh.vertices))
# Compute the surface of the entire complex and from that compute the interface.
v3, f3, _, _, _ = computeMSMS(protonate_file, protonate=True)
# Regularize the mesh
mesh = pymesh.form_mesh(v3, f3)
# I believe It is not necessary to regularize the full mesh. This can speed up things by a lot.
full_regular_mesh = mesh
# Find the vertices that are in the iface.
v3 = full_regular_mesh.vertices
# Find the distance between every vertex in regular_mesh.vertices and those in the full complex.
kdt = KDTree(v3)
d, r = kdt.query(regular_mesh.vertices)
d = np.square(d) # Square d, because this is how it was in the pyflann version.
assert(len(d) == len(regular_mesh.vertices))
iface_v = np.where(d >= 2.0)[0]
iface[iface_v] = 1.0
# Convert to ply and save.
save_ply(ply_filename, regular_mesh.vertices, regular_mesh.faces, normals=vertex_normal, \
        charges=vertex_charges, normalize_charges=True, hbond=vertex_hbond, \
            hphob=vertex_hphobicity, iface=iface)

print(f"Finished extract features from the strucutr {pdb_filename.split('.pdb')[0]}. Thanks for your using!")

# ==========Coumpute Data==========

params = masif_opts['ligand']

output_dict = {} # vertice_number, xyz, neigh_indecies, si, ddc, hbond, charge, hphob, rho, theta

# Compute shape complementarity between the two proteins.
rho = {}
neigh_indices = {}
mask = {}
input_feat = {}
theta = {}
iface_labels = {}
verts = {}

input_feat, rho, theta, mask, neigh_indices, iface_labels, verts, faces = read_data_from_surface(ply_filename, params)

if collapse_type == 'GTA':
    # NDP_points = np.array([[ 1.603, 19.007, 10.355], [-0.716, 19.148, 10.857], [ 4.897, 18.192, 11.585], [ 3.434, 18.433, 11.889],
    #                       [ 4.877, 17.968, 10.078], [ 2.986, 19.226, 10.672], [ 4.610, 16.520,  9.690], [ 0.598, 19.410, 11.266],
    #                       [ 1.254, 18.402,  9.154], [-0.003, 18.158,  8.777], [-1.113, 18.547,  9.672], [ 3.793, 18.777,  9.572],
    #                       [ 5.658, 19.372, 11.844], [ 3.247, 19.141, 13.097], [ 5.595, 16.103,  8.757], [ 7.677, 14.976,  7.985],
    #                       [ 0.831, 19.954, 12.347], [ 7.989, 12.795,  6.698], [ 8.490, 12.923,  9.278], [ 7.015, 14.821, 10.445],
    #                       [ 7.877, 17.085,  9.421], [-2.280, 18.338,  9.355], [ 8.487, 13.560,  7.905], [ 7.110, 15.788,  9.285]])
    # SUGAR_points = np.array([[12.831, 13.363,  7.049], [13.291, 14.698,  6.461], [11.604, 12.848,  6.296], [12.121, 15.688,  6.408],
    #                         [10.510, 13.918,  6.253], [12.497, 16.997,  5.719], [11.014, 15.129,  5.683], [ 9.977, 14.128,  7.560],
    #                         [13.877, 12.400,  6.955], [13.783, 14.492,  5.135], [11.093, 11.677,  6.924], [13.052, 17.872,  6.684]])
    # 2vfz展示
    NDP_points = np.array([
        [5.592  ,16.612  , 8.001],[6.816  ,15.928  , 5.527],[3.479  ,12.039  ,12.290],[4.570  ,14.080  ,13.033],[3.973  ,15.363  ,13.417],[4.432  ,17.575  , 8.080],
        [7.685  ,17.129  , 5.804],[4.564  ,16.635  ,13.189],[3.534  ,13.457  ,12.117],[5.688  ,16.767  ,12.658],[6.980  ,17.066  , 8.374],[6.329  ,15.627  , 4.129],
        [4.009  ,13.803  ,10.725],[3.919  ,17.747  ,13.562],[5.571  ,15.920  , 6.544],[3.675  ,12.764  , 9.803],[2.715  ,17.687  ,14.135],[5.514  ,13.946  ,10.877],
        [2.133  ,18.739  ,14.482],[5.760  ,14.223  ,12.256],[2.113  ,16.459  ,14.363],[6.072  ,15.097  ,10.058],[5.250  ,15.344  , 8.924],[2.764  ,15.300  ,13.979]
    ])
    SUGAR_points = np.array([
        [ 9.885  ,11.033  , 5.073],[10.480  ,12.393  , 4.845],[11.608  ,12.536  , 4.405],[ 9.001  ,12.112  , 9.408],[ 9.915  ,12.481  ,10.442],[ 9.347  ,12.910  , 8.162],
        [ 8.622  ,14.137  , 8.125],[10.824  ,13.268  , 8.144],[11.137  ,13.932  , 9.366],[11.157  ,14.239  , 7.022],[11.552  ,15.485  , 7.610],[ 9.989  ,14.500  , 6.074],
        [ 9.684  ,13.419  , 5.139],[ 8.745  ,14.856  , 6.887],[ 7.597  ,14.640  , 6.076]
    ])


elif collapse_type == 'GTB':
    NDP_points = np.array([[ 0.297,-10.026, -4.534], [ 1.190,-11.657, -5.720], [ 2.245, -9.736, -3.099],
                          [ 4.071,-11.142, -3.509], [ 4.264, -9.568, -1.751], [-0.564, -8.645, -2.688],
                          [-0.287, -7.148, -2.837], [-0.679, -9.071, -4.101], [-1.062, -6.875, -4.110],
                          [-0.659, -5.560, -4.737], [ 1.542,-10.285, -4.082], [ 0.099,-10.888, -5.550],
                          [ 2.123,-11.301, -4.806], [ 3.402,-11.734, -4.516], [ 3.494,-10.141, -2.797],
                          [-0.670, -7.937, -4.909], [-1.792, -8.725, -1.997], [-0.746, -6.425, -1.758],
                          [-1.640, -4.598, -4.574], [-1.563, -0.069, -2.051], [-0.582, -2.255, -3.484],
                          [ 0.211, -3.048, -5.970], [-2.242, -2.264, -5.503], [ 0.374,  0.232, -4.051],
                          [-2.079, -0.374, -4.565], [ 3.932,-12.663, -5.180], [-1.092, -3.046, -4.863],
                          [-0.993, -0.598, -3.527]])
    SUGAR_points = np.array([[-4.035,  0.768, -0.403], [-3.632,  0.106,  0.906], [-3.843, -0.224, -1.509],
                            [-2.203, -0.359,  0.735], [-2.511, -0.911, -1.485], [-1.687, -0.971,  2.039],
                            [-2.136, -1.352, -0.224], [-1.563, -0.069, -2.051], [-5.384,  1.128, -0.393],
                            [-3.642,  1.071,  1.926], [-4.888, -1.150, -1.576], [-0.922, -2.091,  1.736]])

distances = np.full((verts.shape[0],), False, dtype=bool)
for point in NDP_points:
   distances_temp = np.sqrt(np.sum((verts - point) ** 2, axis=1))
   distances_temp = distances_temp < sample_redies_udp
   distances = distances | distances_temp
for point in SUGAR_points:
   distances_temp = np.sqrt(np.sum((verts - point) ** 2, axis=1))
   distances_temp = distances_temp < sample_redies_sugar
   distances = distances | distances_temp

def clean_mesh(vertices, edges, component_threshold=3):
    """清洗孤立点和小组件"""
    n = len(vertices)

    # 构建邻接表
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    # 查找连通组件
    visited = [False] * n
    components = []
    for i in range(n):
        if not visited[i]:
            component = []
            stack = [i]
            visited[i] = True
            while stack:
                node = stack.pop()
                component.append(node)
                for neighbor in adj[node]:
                    if not visited[neighbor]:
                        visited[neighbor] = True
                        stack.append(neighbor)
            components.append(component)

    # 保留较大组件
    valid = set()
    for comp in components:
        if len(comp) >= component_threshold:
            valid.update(comp)

    # 迭代删除度数<=1的顶点
    current_valid = valid.copy()
    while True:
        degrees = {u: len([v for v in adj[u] if v in current_valid]) for u in current_valid}
        to_remove = {u for u, d in degrees.items() if d <= 1}
        if not to_remove:
            break
        current_valid -= to_remove

    # 生成清洗掩码
    mask_clean = np.zeros(n, dtype=bool)
    mask_clean[list(current_valid)] = True
    return mask_clean

# ==================== 获取边的信息 ====================
# 创建一个空的边集来存储边，使用集合来避免重复边
edges = set()
# 遍历每个面，将其顶点连接成边
for face in faces:
    # 获取每个面的三条边，顶点索引两两组合
    edges.add(tuple(sorted([face[0], face[1]])))
    edges.add(tuple(sorted([face[1], face[2]])))
    edges.add(tuple(sorted([face[2], face[0]])))
# 将边转换为numpy数组
edges = np.array(list(edges))

# 使用np.where获取值为True的元素的索引
true_indices = np.where(distances)[0]
# 创建一个从0开始的索引列表，这里的长度与true_indices相同
new_indices = np.arange(len(true_indices))
# 创建映射关系，将原始索引映射到新的索引
index_mapping = {original_index: new_index for original_index, new_index in zip(true_indices, new_indices)}

# 拿到新索引的边
local_edge = []
for e in edges:
    if e[0] in true_indices and e[1] in true_indices:
        local_edge.append([index_mapping[e[0]], index_mapping[e[1]]])

# output_dict['vertice_number'] = int(verts.shape[0])
# output_dict['index_mapping'] = index_mapping
# output_dict['distances'] = distances
output_dict['xyz'] = verts[distances, :]
output_dict['edges'] = np.array(local_edge)
# output_dict['neigh_indecies'] = neigh_indices
output_dict['si'] = input_feat[:, :, 0][distances, :]
# output_dict['ddc'] = input_feat[:, :, 1][distances, :]
output_dict['hbond'] = input_feat[:, :, 2][distances, :]
output_dict['charge'] = input_feat[:, :, 3][distances, :]
output_dict['hphob'] = input_feat[:, :, 4][distances, :]
# output_dict['rho'] = rho[distances, :]
# output_dict['theta'] = theta[distances, :]

# ==================== 网格清洗步骤 ====================
vertices_sampled = output_dict['xyz']
edges_sampled = output_dict['edges']
mask_clean = clean_mesh(vertices_sampled, edges_sampled)

# 更新output_dict中的属性
output_dict['xyz'] = output_dict['xyz'][mask_clean]
output_dict['edges'] = np.array([[u, v] for u, v in edges_sampled if mask_clean[u] and mask_clean[v]])

# 重新映射索引
true_indices_clean = np.where(mask_clean)[0]
index_mapping_clean = {old: new for new, old in enumerate(true_indices_clean)}
output_dict['edges'] = np.array([[index_mapping_clean[u], index_mapping_clean[v]] for u, v in output_dict['edges']])

# 更新其他特征数据
# for key in ['si', 'ddc', 'hbond', 'charge', 'hphob', 'rho', 'theta']:
for key in ['si', 'hbond', 'charge', 'hphob']:
    output_dict[key] = output_dict[key][mask_clean]
    output_dict[key] = output_dict[key][:,0:1]

# 更新index_mapping
# original_to_sampled = output_dict['index_mapping']
# sampled_to_clean = {old: idx for idx, old in enumerate(true_indices_clean)}
# output_dict['index_mapping'] = {orig: sampled_to_clean[sampled] for orig, sampled in original_to_sampled.items() if sampled in sampled_to_clean}


# Save data only if everything went well. 
# np.save(storage_filename, output_dict)
print(f"Finished extract features from the strucutr {pdb_filename.split('.pdb')[0]}. Thanks for your using!")



Finished extract features from the strucutr 2vfz_align. Thanks for your using!
Finished extract features from the strucutr 2vfz_align. Thanks for your using!


In [5]:
# 可视化结构，三个，分别是原始的结构、fixed后的结构、local site后的结构。
import os

def write_vertices_to_pdb(filename, vertices):
    f = open(filename, 'w')
    step = 0
    for v in vertices:
        step += 1
        line = ""
        line = line + f"{'ATOM':<6}"
        line = line + f"{step:>5}"
        line = line + f"  {'C':<4}"
        line = line + f"{'ALA':<4}"
        line = line + f"{'A':<1}"
        line = line + f"{1:>4}"
        line = line + f"{round(v[0], 3):>12}"
        line = line + f"{round(v[1], 3):>8}"
        line = line + f"{round(v[2], 3):>8}"
        line = line + f"{1.00:>6}"
        line = line + f"{95.00:>6}" # B-factor, 根据内容进行设置
        line = line + f"{'C':>12}"
        line = line + f"{'  ':>2}\n"
        f.write(line)
    f.close()

def write_vertices_to_pdb_feature(filename, output_dict, feature=None):
    f = open(filename, 'w')
    step = 0
    vertices = output_dict['xyz']
    features = output_dict[feature] if feature is not None else None
    for idx, v in enumerate(vertices):
        step += 1
        line = ""
        line = line + f"{'ATOM':<6}"
        line = line + f"{step:>5}"
        line = line + f"  {'C':<4}"
        line = line + f"{'ALA':<4}"
        line = line + f"{'A':<1}"
        line = line + f"{1:>4}"
        line = line + f"{round(v[0], 3):>12}"
        line = line + f"{round(v[1], 3):>8}"
        line = line + f"{round(v[2], 3):>8}"
        line = line + f"{1.00:>6}"
        if feature is not None:
            line = line + f"{features[idx][0]*100:>6.2f}"
        else:
            line = line + f"{95.00:>6}" # B-factor, 根据内容进行设置
        line = line + f"{'C':>12}"
        line = line + f"{'  ':>2}\n"
        f.write(line)
    f.close()

storage_path = './diffpool/predict_data/'
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_original.pdb'), mesh_original.vertices)
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_fixed.pdb'), mesh_fixed.vertices)
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_local.pdb'), verts[distances, :])

# 展示活性架构表面景观
storage_path = './ref/'
feature = 'hphob' # si, hbond, charge, hphob
write_vertices_to_pdb_feature(os.path.join(storage_path, f'grid_local_{feature}.pdb'), output_dict, feature=feature)